# 4.11 Python

Chapter 4 introduces three distributions that arise from sequences of independent Bernoulli trials: the Geometric, the Negative Binomial, and the Poisson. Python's `scipy.stats` module provides each through a uniform interface of `.pmf`, `.cdf`, and related methods. This section also uses simulation to verify two expectation results from the chapter — the expected number of matches when a shuffled deck is compared with the original ordering, and the expected number of distinct birthdays in a group of people.

In [ ]:
import numpy as np
import scipy.stats as st
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(palette="deep")
rng = np.random.default_rng(seed=42)

## Geometric Distribution

If $X \sim \mathrm{Geom}(p)$, then $X$ counts the number of failures before the first success, with PMF

$$P(X = k) = (1-p)^k\, p, \quad k = 0, 1, 2, \ldots$$

`scipy.stats` offers two related objects for this family. `st.nbinom(n, p)` models the number of failures before $n$ successes; setting $n = 1$ gives exactly $\mathrm{Geom}(p)$. The separate `st.geom(p)` models the *first-success* (FS) distribution, which counts total trials including the success and starts at 1 instead of 0.

To compute $P(X = 3)$ and $P(X \le 3)$ where $X \sim \mathrm{Geom}(0.5)$:

In [ ]:
p = 0.5
print(st.nbinom.pmf(3, 1, p))   # P(X = 3)
print(st.nbinom.cdf(3, 1, p))   # P(X <= 3)

$P(X = 3) = (0.5)^3 \cdot 0.5 = 0.0625$, and the CDF sums the PMF from 0 through 3, giving $1 - (0.5)^4 = 0.9375$.

Plotting the PMF over the first several values shows the characteristic geometric decay:

In [ ]:
k = np.arange(0, 10)
pmf_geom = st.nbinom.pmf(k, 1, p)

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=k, y=pmf_geom, ax=ax)
ax.set_xlabel("k  (failures)")
ax.set_ylabel("P(X = k)")
ax.set_title(f"Geometric PMF  (p = {p})")
plt.tight_layout()
plt.show()

To generate random samples, `rng.geometric(p, size=N)` produces $N$ draws from the first-success distribution — the number of trials to the first success. Subtracting 1 converts to the failure count matching $\mathrm{Geom}(p)$:

In [ ]:
geom_samples = rng.geometric(0.8, size=100) - 1   # Geom(0.8): number of failures
fs_samples   = rng.geometric(0.8, size=100)        # FS(0.8):   number of trials
print("Geom(0.8):", geom_samples[:10])
print("FS(0.8):  ", fs_samples[:10])

## Negative Binomial Distribution

$X \sim \mathrm{NBin}(r, p)$ counts the number of failures before the $r$-th success, generalising the Geometric. Its PMF is

$$P(X = k) = \binom{k + r - 1}{k}\, p^r (1-p)^k, \quad k = 0, 1, 2, \ldots$$

`st.nbinom(r, p)` uses this same parameterisation. To find the $\mathrm{NBin}(5, 0.5)$ PMF at $k = 3$:

In [ ]:
st.nbinom.pmf(3, 5, 0.5)   # P(X = 3) for NBin(5, 0.5)

## Poisson Distribution

If $X \sim \mathrm{Pois}(\lambda)$, the PMF is

$$P(X = k) = \frac{e^{-\lambda}\, \lambda^k}{k!}, \quad k = 0, 1, 2, \ldots$$

`st.poisson(lam)` handles both PMF and CDF calculations. To find $P(X \le 2)$ where $X \sim \mathrm{Pois}(10)$:

In [ ]:
st.poisson.cdf(2, 10)   # P(X <= 2) for Pois(10)

## Matching Simulation

Example 4.4.4 asks for the expected number of matches when a shuffled deck of $n$ cards is compared position-by-position with the original ordering — a match at position $i$ means the shuffled card in slot $i$ is the same as the original card $i$. Using indicator random variables, the expected number of matches is exactly 1, regardless of deck size.

We verify this by simulation. `rng.permutation(n)` produces a random arrangement of $0, 1, \ldots, n-1$; a match at position $i$ means the permuted value equals $i$, the same as its 0-based index. Repeating $10^4$ times and averaging:

In [ ]:
n = 100
r = [np.sum(rng.permutation(n) == np.arange(n)) for _ in range(10**4)]
np.mean(r)

The result is very close to 1. You can substitute any value of $n$ and `np.mean(r)` will remain near 1.

## Distinct Birthdays Simulation

Example 4.4.5 computes the expected number of distinct birthdays among $k$ people, each born on one of 365 days chosen uniformly at random. By the indicator r.v. approach — one indicator per calendar day — the expected count is

$$365\left(1 - \left(\frac{364}{365}\right)^k\right).$$

For $k = 20$, we simulate $10^4$ groups. `rng.integers(1, 366, size=k)` draws $k$ birthdays uniformly from 1 through 365, and `len(np.unique(...))` counts how many are distinct:

In [ ]:
k = 20
r = [len(np.unique(rng.integers(1, 366, size=k))) for _ in range(10**4)]
print("Simulated mean:    ", np.mean(r))
print("Theoretical value: ", 365 * (1 - (364 / 365)**k))

Both values are approximately 19.5. Changing $k$ to any other value produces equally close agreement between the simulation and the formula.